# Homework 07: Outliers and Risk Assumptions

Detects outliers with IQR and Z-score, winsorizes as an alternative to dropping, and runs a
sensitivity analysis (summary stats and a simple regression) comparing all/filtered/winsorized
on a synthetic daily-return series with five injected shocks in May 2022.

## Setup: Generate the Dataset
Business-day returns with five large shocks injected in May, plus a correlated second column.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

raw_dir = 'data/raw'
processed_dir = 'data/processed'
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

dates = pd.date_range(start="2022-01-03", end="2022-06-10", freq="B")
np.random.seed(17)

returns = np.random.normal(0, 0.01, size=len(dates))
mask_pre_may = dates < "2022-05-01"
returns[mask_pre_may] -= 0.0015

shock_values = {
    "2022-05-02": 0.1748425237194541,
    "2022-05-03": -0.16825801732486943,
    "2022-05-06": -0.19667220757153227,
    "2022-05-09": 0.21240223590614747,
    "2022-05-12": -0.178729287231294,
}
for d, v in shock_values.items():
    idx = np.where(dates == pd.to_datetime(d))[0][0]
    returns[idx] = v

daily_return_2 = returns * 0.6 + np.random.normal(0, 0.005, size=len(dates))

df = pd.DataFrame({"date": dates, "daily_return": returns, "daily_return_2": daily_return_2})

csv_path = os.path.join(raw_dir, 'outliers_homework.csv')
df.to_csv(csv_path, index=False)
print(f'Saved {csv_path}')

Saved data/raw\outliers_homework.csv


## Load Data and Outlier Functions
`detect_outliers_iqr`, `detect_outliers_zscore`, and `winsorize_series` live in `src/outliers.py`.

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from src.outliers import detect_outliers_iqr, detect_outliers_zscore, winsorize_series

df = pd.read_csv(csv_path)
df.head()

,date,daily_return,daily_return_2
0,2022-01-03,0.001263,0.003834
1,2022-01-04,-0.020046,-0.009506
2,2022-01-05,0.004739,-0.000535
3,2022-01-06,0.009953,0.012539
4,2022-01-07,0.008872,0.009840


## Apply Detection and Create Flags

In [3]:
target_col = 'daily_return'
df['outlier_iqr'] = detect_outliers_iqr(df[target_col], k=1.5)
df['outlier_z'] = detect_outliers_zscore(df[target_col], threshold=3.0)

print('Flagged by IQR:', df['outlier_iqr'].sum(), 'of', len(df))
print('Flagged by Z-score:', df['outlier_z'].sum(), 'of', len(df))
df.loc[df['outlier_iqr'] | df['outlier_z'], ['date', 'daily_return', 'outlier_iqr', 'outlier_z']]

Flagged by IQR: 9 of 115
Flagged by Z-score: 5 of 115


,date,daily_return,outlier_iqr,outlier_z
41,2022-03-01,0.031952,True,False
63,2022-03-31,-0.031101,True,False
85,2022-05-02,0.174843,True,True
86,2022-05-03,-0.168258,True,True
87,2022-05-04,-0.033999,True,False
89,2022-05-06,-0.196672,True,True
90,2022-05-09,0.212402,True,True
92,2022-05-11,0.028711,True,False
93,2022-05-12,-0.178729,True,True


### Visual Checks

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].boxplot(df[target_col])
axes[0].set_title(f'Boxplot: {target_col}')
axes[1].hist(df[target_col], bins=30)
axes[1].set_title(f'Histogram: {target_col}')
fig.tight_layout()
fig.savefig('data/processed/outlier_diagnostics.png', dpi=120)
print('Saved data/processed/outlier_diagnostics.png')

Saved data/processed/outlier_diagnostics.png


## Sensitivity Analysis: Summary Statistics
All rows, IQR-filtered, and winsorized.

In [5]:
summ_all = df[target_col].describe()[['mean', '50%', 'std']].rename({'50%': 'median'})
summ_filtered = df.loc[~df['outlier_iqr'], target_col].describe()[['mean', '50%', 'std']].rename({'50%': 'median'})
winsorized = winsorize_series(df[target_col])
summ_winsorized = winsorized.describe()[['mean', '50%', 'std']].rename({'50%': 'median'})

comp_stats = pd.concat({'all': summ_all, 'filtered_iqr': summ_filtered, 'winsorized': summ_winsorized}, axis=1)
comp_stats

,all,filtered_iqr,winsorized
mean,-0.001434,-0.000039,-0.000251
median,-0.000187,-0.000100,-0.000187
std,0.040579,0.009443,0.010623


## Sensitivity Analysis: Regression
`daily_return_2` regressed on `daily_return`, all vs. IQR-filtered vs. winsorized.

In [6]:
resp_col = 'daily_return_2'
keep = ~df['outlier_iqr']

X_all = df[[target_col]].to_numpy(); y_all = df[resp_col].to_numpy()
X_filtered = df.loc[keep, [target_col]].to_numpy(); y_filtered = df.loc[keep, resp_col].to_numpy()
X_wins = winsorized.to_numpy().reshape(-1, 1)
y_wins = winsorize_series(df[resp_col]).to_numpy()

results = {}
for name, X, y in [('all', X_all, y_all), ('filtered_iqr', X_filtered, y_filtered), ('winsorized', X_wins, y_wins)]:
    model = LinearRegression().fit(X, y)
    results[name] = {
        'slope': model.coef_[0],
        'intercept': model.intercept_,
        'r2': model.score(X, y),
        'mae': mean_absolute_error(y, model.predict(X)),
    }

results_df = pd.DataFrame(results).T
results_df

,slope,intercept,r2,mae
all,0.605869,0.000201,0.961859,0.003951
filtered_iqr,0.589679,-0.000049,0.573566,0.003851
winsorized,0.649705,-0.000017,0.689444,0.003741


## Reflection

**Method and thresholds.** IQR with k=1.5 is the primary detector: it doesn't assume normality
the way a Z-score does, and `daily_return` has five deliberately injected shocks that are far
enough from the bulk of the data that a reasonable detector should catch them. Z-score at
threshold=3 is run alongside it as a check. Winsorizing at the 5th/95th percentile is used as a
middle ground that keeps every row instead of dropping any of them outright.

**Assumptions.** IQR assumes the middle 50% of the data is a fair description of "normal"
behavior. Z-score assumes something closer to a normal distribution, and the five shocks push
against that assumption themselves: they fatten the tails and inflate the sample standard
deviation, which raises the bar a point has to clear to be flagged.

**Observed impact.** The two methods do not agree. Z-score flags exactly the 5 injected shock
days and nothing else, out of 115 rows. IQR flags 9: the same 5 shocks plus 4 ordinary trading
days (two in March, two right next to the May shocks) that aren't shocks at all, just points
that happen to sit past a 1.5x-IQR fence. That's the inflated-sigma effect working in Z-score's
favor here: the shocks widen `sigma` enough that only the shocks themselves clear 3 standard
deviations, while IQR's quartile-based fence doesn't move the same way and ends up stricter.
Dropping IQR's 9 flagged points shrinks the standard deviation more than dropping Z-score's 5,
and changes the regression slope and R^2 between `daily_return` and `daily_return_2` by more, for
the same reason: it's discarding four extra points of real, non-shock variation along with the
shocks. Winsorizing lands between the three, since it dampens every flagged point's influence
without deleting any trading day outright.

**Risks if these assumptions are wrong.** If a "shock" here were actually a real regime change
(a genuine repricing event) rather than noise, dropping or winsorizing it would erase the exact
signal a monitor like this project is trying to catch. That's the central risk this stage exists
to name: an outlier filter tuned for noise will also suppress the rare, real events the whole
project cares about, and here IQR's extra 4 false flags show that risk concretely, a stricter
detector doesn't just catch more real outliers, it can also catch ordinary days that shouldn't
be touched. Any threshold chosen here needs to be revisited once it's applied to real market
data instead of a synthetic series with outliers built in on purpose.